In [1]:
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
from datasets import load_dataset

model_name = 'bert-base-uncased'
dataset = load_dataset('glue', 'sst2')
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

print(dataset)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})


In [2]:
def tokenize_funciton(example):
    return tokenizer(
        example['sentence'],
        truncation=True,
        max_length=128,
    )

tokenized = dataset.map(tokenize_funciton, batched=True)
print(tokenized['train'].column_names)
print(tokenized['train'][0])

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

['sentence', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask']
{'sentence': 'hide new secretions from the parental units ', 'label': 0, 'idx': 0, 'input_ids': [101, 5342, 2047, 3595, 8496, 2013, 1996, 18643, 3197, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [3]:
tokenized = tokenized.remove_columns(['sentence', 'idx'])
tokenized = tokenized.rename_column('label', 'labels')
print(tokenized['train'].column_names)

['labels', 'input_ids', 'token_type_ids', 'attention_mask']


In [4]:
print(tokenized['train'][0])

{'labels': 0, 'input_ids': [101, 5342, 2047, 3595, 8496, 2013, 1996, 18643, 3197, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [5]:
tokenized['train'].set_format('torch', columns=['input_ids', 'labels', 'attention_mask'])
tokenized['validation'].set_format('torch', columns=['input_ids', 'labels', 'attention_mask'])
print(tokenized['train'][0])

{'labels': tensor(0), 'input_ids': tensor([  101,  5342,  2047,  3595,  8496,  2013,  1996, 18643,  3197,   102]), 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1])}


In [6]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

train_loader = DataLoader(
    tokenized['train'],
    batch_size=32,
    shuffle=True,
    collate_fn=data_collator,
)

eval_loader = DataLoader(
    tokenized['validation'],
    batch_size=64,
    shuffle=False,
    collate_fn=data_collator,
)

batch = next(iter(train_loader))
print(batch.keys())
print(batch['input_ids'].shape)
print(batch)


KeysView({'labels': tensor([1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0,
        1, 0, 1, 0, 1, 0, 1, 1]), 'input_ids': tensor([[  101,  1037, 20934,  ...,     0,     0,     0],
        [  101,  3268,  2091,  ...,     0,     0,     0],
        [  101,  2709,  2000,  ...,     0,     0,     0],
        ...,
        [  101,  1037, 13544,  ...,     0,     0,     0],
        [  101,  1997,  2216,  ...,     0,     0,     0],
        [  101,  2007,  2635,  ...,     0,     0,     0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])})
torch.Size([32, 43])
{'labels': tensor([1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0,
        1, 0, 1, 0, 1, 0, 1, 1]), 'input_ids': tensor([[  101,  1037, 20934,  ...,     0,     0,     0],
        [  101,  3268,  2091,  ...,

In [24]:
from transformers import AutoModelForSequenceClassification
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(type(device))
print(device)

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

num_epochs = 3
num_training_steps = num_epochs * len(train_loader)
num_warmup_steps = int(0.06*num_training_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_training_steps=num_training_steps,
    num_warmup_steps=num_warmup_steps,
)

model.train()
for epoch in range(num_epochs):
    total_loss = 0
    for step, batch in enumerate(train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()

        torch.nn.utils.clip_grad_norm(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        optimizer.zero_grad()

        total_loss += loss.item()

        if step % 200 == 0:
            print(f"Epoch {epoch+1}, Step {step}, Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} average loss: {avg_loss:.4f}\n")

<class 'torch.device'>
cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/tmp/ipykernel_416/1276259167.py:33: 

Epoch 1, Step 0, Loss: 0.6413
Epoch 1, Step 200, Loss: 0.1042
Epoch 1, Step 400, Loss: 0.2941
Epoch 1, Step 600, Loss: 0.1929
Epoch 1, Step 800, Loss: 0.2280
Epoch 1, Step 1000, Loss: 0.3338
Epoch 1, Step 1200, Loss: 0.2422
Epoch 1, Step 1400, Loss: 0.3314
Epoch 1, Step 1600, Loss: 0.0888
Epoch 1, Step 1800, Loss: 0.2165
Epoch 1, Step 2000, Loss: 0.1689
Epoch 1 average loss: 0.2411

Epoch 2, Step 0, Loss: 0.1602
Epoch 2, Step 200, Loss: 0.0200
Epoch 2, Step 400, Loss: 0.0182
Epoch 2, Step 600, Loss: 0.0628
Epoch 2, Step 800, Loss: 0.1872
Epoch 2, Step 1000, Loss: 0.2597
Epoch 2, Step 1200, Loss: 0.1273
Epoch 2, Step 1400, Loss: 0.0134
Epoch 2, Step 1600, Loss: 0.0516
Epoch 2, Step 1800, Loss: 0.0415
Epoch 2, Step 2000, Loss: 0.0606
Epoch 2 average loss: 0.1136

Epoch 3, Step 0, Loss: 0.1679
Epoch 3, Step 200, Loss: 0.0079
Epoch 3, Step 400, Loss: 0.0075
Epoch 3, Step 600, Loss: 0.0395
Epoch 3, Step 800, Loss: 0.0062
Epoch 3, Step 1000, Loss: 0.0122
Epoch 3, Step 1200, Loss: 0.1918
Epoc

In [25]:
def evaluate_model(model, eval_loader, device):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in eval_loader:
            batch = {k: v.to(device) for k, v in batch.items()}

            outputs = model(**batch)
            logits = outputs.logits
            preds = torch.argmax(logits, dim=-1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(batch['labels'].cpu().numpy())

    accuracy = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
    model.train()
    return accuracy

acc = evaluate_model(model, eval_loader, device)
print(f"Validation Accuracy: {acc:.4f}")



Validation Accuracy: 0.9289


In [7]:
from transformers import AutoModelForSequenceClassification
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(type(device))
print(device)

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

num_epochs = 3
num_training_steps = num_epochs * len(train_loader)
num_warmup_steps = int(0.06*num_training_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_training_steps=num_training_steps,
    num_warmup_steps=num_warmup_steps,
)


def evaluate_model(model, eval_loader, device):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in eval_loader:
            batch = {k: v.to(device) for k, v in batch.items()}

            outputs = model(**batch)
            logits = outputs.logits
            preds = torch.argmax(logits, dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(batch['labels'].cpu().numpy())

    accuracy = sum(p == l for p, l in zip(all_preds, all_labels))/len(all_labels)
    return accuracy


best_acc = 0

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for step, batch in enumerate(train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        total_loss += loss.item()

        if step % 200 == 0:
            print(f"Epoch {epoch + 1}, Step {step}, Train Loss {loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)
    acc = evaluate_model(model, eval_loader, device)
    print(f"Epoch {epoch+1} finishes, Average Train Dataset Loss {avg_loss:.4f}, Validation Dataset Accuracy {acc:.4f}")

    if acc > best_acc:
        best_acc = acc
        model.save_pretrained('./best-sst2-bert')
        tokenizer.save_pretrained('./best-sst2-bert')
        print(f"  → Saved best model (acc={acc:.4f})")

print(f"\nBest validation accuracy: {best_acc:.4f}")


<class 'torch.device'>
cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1, Step 0, Train Loss 0.7133
Epoch 1, Step 200, Train Loss 0.3143
Epoch 1, Step 400, Train Loss 0.1862
Epoch 1, Step 600, Train Loss 0.0717
Epoch 1, Step 800, Train Loss 0.2328
Epoch 1, Step 1000, Train Loss 0.0978
Epoch 1, Step 1200, Train Loss 0.0910
Epoch 1, Step 1400, Train Loss 0.1725
Epoch 1, Step 1600, Train Loss 0.1802
Epoch 1, Step 1800, Train Loss 0.0838
Epoch 1, Step 2000, Train Loss 0.2607
Epoch 1 finishes, Average Train Dataset Loss 0.2383, Validation Dataset Accuracy 0.9278


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Saved best model (acc=0.9278)
Epoch 2, Step 0, Train Loss 0.2799
Epoch 2, Step 200, Train Loss 0.1731
Epoch 2, Step 400, Train Loss 0.1548
Epoch 2, Step 600, Train Loss 0.0157
Epoch 2, Step 800, Train Loss 0.0448
Epoch 2, Step 1000, Train Loss 0.1609
Epoch 2, Step 1200, Train Loss 0.0244
Epoch 2, Step 1400, Train Loss 0.0578
Epoch 2, Step 1600, Train Loss 0.1689
Epoch 2, Step 1800, Train Loss 0.1388
Epoch 2, Step 2000, Train Loss 0.3104
Epoch 2 finishes, Average Train Dataset Loss 0.1104, Validation Dataset Accuracy 0.9220
Epoch 3, Step 0, Train Loss 0.0711
Epoch 3, Step 200, Train Loss 0.0113
Epoch 3, Step 400, Train Loss 0.1250
Epoch 3, Step 600, Train Loss 0.0035
Epoch 3, Step 800, Train Loss 0.0210
Epoch 3, Step 1000, Train Loss 0.1093
Epoch 3, Step 1200, Train Loss 0.0770
Epoch 3, Step 1400, Train Loss 0.0037
Epoch 3, Step 1600, Train Loss 0.0093
Epoch 3, Step 1800, Train Loss 0.0391
Epoch 3, Step 2000, Train Loss 0.1478
Epoch 3 finishes, Average Train Dataset Loss 0.0693, Val

In [9]:
best_model = AutoModelForSequenceClassification.from_pretrained('./best-sst2-bert').to(device)
tokenizer = AutoTokenizer.from_pretrained("./best-sst2-bert")

def predict(text, model, tokenizer, device):
    model.eval()
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=128)
    inputs = {k:v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
    pred = torch.argmax(logits, dim=-1).item()
    probs = torch.softmax(logits, dim=-1)
    return "positive" if pred == 1 else "negative", probs[0].cpu().numpy()

label, probs = predict("This movie was absolutely fantastic!", best_model, tokenizer, device)
print(f"Prediction: {label}, Probs: {probs}")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Prediction: positive, Probs: [0.00108531 0.9989147 ]
